# Notebook 11: Multi-Dimensional Multi-Objective Pareto Frontier Analysis

**CPU-only / GPU-agnostic, Google Drive independent, derived analysis.**

## Research Motivation & Multi-Objective Engineering Context
Deploying Small Language Models (SLMs) on edge-class accelerators (such as an NVIDIA T4 GPU) is a multi-objective engineering problem with conflicting constraints. Optimizing a single metric in isolation (e.g. energy alone) can result in operationally useless configurations (e.g., severe retrieval failure). A complete systems evaluation requires optimizing across **four simultaneous dimensions**:

1. **Minimize Total Energy** ($E$, Joules at 4096 tokens) $\downarrow$
2. **Minimize Persistent KV Cache Memory** ($M$, MiB) $\downarrow$
3. **Maximize Decode Throughput** ($T$, tokens/second) $\uparrow$
4. **Maximize Hard-Retrieval Accuracy** ($Q$, exact-match pass rate across 60 calibrated scenarios) $\uparrow$

## Mathematical Dominance & Pareto Optimality Formulation
In a $D$-dimensional objective space with objective vector $\mathbf{y} = (E, M, -T, -Q)$, a compression configuration $A$ strictly dominates configuration $B$ ($A \succ B$) if and only if:
$$\forall k \in \{1,2,3,4\},\; y_{A,k} \le y_{B,k} \quad \text{and} \quad \exists k \in \{1,2,3,4\},\; y_{A,k} < y_{B,k}$$
A configuration is **4D Pareto-optimal** if no other configuration in the design space dominates it.

## Execution Modes
- **Google Colab**: Downloads the public CSE407 project ZIP via `gdown`, extracts validated provisional synthesis sources, executes 4D dominance and bootstrap analysis, and exports all figures, LaTeX tables, and a downloadable ZIP in `/content/`.
- **Local Environment**: Automatically detects local workspace paths and runs seamlessly without external downloads.

In [ ]:
# Run 0: Environment setup, dependencies, and repository acquisition
%pip install -q gdown matplotlib pandas numpy plotly

import os
import sys
import json
import math
import shutil
import zipfile
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

# Set publication-quality matplotlib typography
plt.rcParams.update({
    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8.5,
    "figure.titlesize": 12,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "sans-serif",
})

# Path discovery: check local repository first, then Colab gdown
LOCAL_CANDIDATE = Path("evidence/provisional/bundles/CSE407_cross_model_synthesis_local_provisional")
PARENT_CANDIDATE = Path("../../evidence/provisional/bundles/CSE407_cross_model_synthesis_local_provisional")

if LOCAL_CANDIDATE.is_dir() and (LOCAL_CANDIDATE / "final_integrity.json").is_file():
    SOURCE_DIR = LOCAL_CANDIDATE.resolve()
    OUTPUT_DIR = Path("evidence/accepted/CSE407_multidim_pareto_v1").resolve()
elif PARENT_CANDIDATE.is_dir() and (PARENT_CANDIDATE / "final_integrity.json").is_file():
    SOURCE_DIR = PARENT_CANDIDATE.resolve()
    OUTPUT_DIR = Path("../../evidence/accepted/CSE407_multidim_pareto_v1").resolve()
else:
    # Colab environment: download project archive with gdown
    import gdown
    DRIVE_FILE_ID = "1MtCVx8ZkSVVgQ6Q1SNNEOMnrqvh393O4"
    ZIP_PATH = Path("/content/cse407_project.zip")
    EXTRACT_ROOT = Path("/content/cse407_uploaded_repo")
    OUTPUT_DIR = Path("/content/CSE407_multidim_pareto_v1")
    
    if not ZIP_PATH.exists():
        print("Downloading project archive via gdown...")
        gdown.download(id=DRIVE_FILE_ID, output=str(ZIP_PATH), quiet=False)
    
    if not EXTRACT_ROOT.exists():
        EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(ZIP_PATH) as archive:
            archive.extractall(EXTRACT_ROOT)
            
    matches = [p for p in EXTRACT_ROOT.rglob("CSE407_cross_model_synthesis_local_provisional") if p.is_dir() and (p / "final_integrity.json").is_file()]
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly 1 provisional synthesis bundle, found: {matches}")
    SOURCE_DIR = matches[0].resolve()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Source Bundle: {SOURCE_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")
assert (SOURCE_DIR / "empirical_pareto.csv").is_file(), "Missing empirical_pareto.csv"
assert (SOURCE_DIR / "normalized_energy_rows.csv").is_file(), "Missing normalized_energy_rows.csv"
assert (SOURCE_DIR / "normalized_quality_rows.csv").is_file(), "Missing normalized_quality_rows.csv"
print("Preflight check passed successfully.")

In [ ]:
# Run 1: Load validated datasets and verify 4D multi-objective schema integrity
empirical_pareto = pd.read_csv(SOURCE_DIR / "empirical_pareto.csv")
norm_energy = pd.read_csv(SOURCE_DIR / "normalized_energy_rows.csv")
norm_quality = pd.read_csv(SOURCE_DIR / "normalized_quality_rows.csv")

# Assertions on source evidence integrity
assert len(empirical_pareto) == 10, f"Expected 10 pareto rows, got {len(empirical_pareto)}"
assert empirical_pareto.evidence_tier.eq("measured_system").all()

# Filter raw 4096-token systems energy and hard-60 retrieval rows
e4096 = norm_energy[(norm_energy.context == 4096) & (norm_energy.regime == "raw") & (norm_energy.structural_pass == True)].copy()
q4096 = norm_quality[(norm_quality.quality_population == "hard60") & (norm_quality.structural_pass == True)].copy()

# Extract decode throughput statistics (tokens/second)
tp_stats = e4096.groupby(["model", "normalized_role"])["decode_tok_s"].agg(["mean", "std"]).reset_index()
tp_stats.rename(columns={"mean": "tok_s_mean", "std": "tok_s_std"}, inplace=True)

# Merge with empirical pareto table
df = pd.merge(empirical_pareto, tp_stats, on=["model", "normalized_role"], how="left")

# Map canonical roles and styling
ROLE_SHORT = {
    "fp16": "FP16",
    "awq_wrapper": "AWQ",
    "awq_eviction": "Eviction",
    "awq_tq_ref": "TQ-Ref",
    "awq_eviction_tq_ref": "Eviction + TQ-Ref"
}
ROLE_COLORS = {
    "FP16": "#4A6FA5",
    "AWQ": "#47B8B8",
    "Eviction": "#E67E22",
    "TQ-Ref": "#8E44AD",
    "Eviction + TQ-Ref": "#2ECC71"
}
MODEL_TITLES = {
    "llama": "Llama 3.2-3B (3:1 GQA)",
    "qwen": "Qwen2.5-3B (8:1 GQA)"
}

df["role_name"] = df["normalized_role"].map(ROLE_SHORT)
df["cache_mib"] = df["persistent_kv_bytes_mean"] / (1024 * 1024)

print("4D Multi-Objective Measurement Summary at 4096 Tokens:")
print(df[["model", "source_variant", "role_name", "energy_mean_j", "cache_mib", "tok_s_mean", "retrieval_estimate"]])

In [ ]:
# Run 2: 4D Multi-Objective Pareto Dominance and 2000-Iteration Bootstrap Engine

def is_pareto_efficient_4d(costs):
    """
    Vectorized non-dominated sorting in D dimensions.
    costs: (N, D) array where LOWER values represent BETTER outcomes for all columns.
    Returns boolean array of length N where True indicates non-dominated Pareto optimality.
    """
    is_eff = np.ones(costs.shape[0], dtype=bool)
    for i, c in enumerate(costs):
        if is_eff[i]:
            # Point i is strictly dominated if any other point is <= on all dimensions and < on at least one
            dominated = np.any((costs <= c).all(axis=1) & (costs < c).any(axis=1))
            if dominated:
                is_eff[i] = False
    return is_eff

# Compute 4D Pareto optimality on sample means
for model in ["llama", "qwen"]:
    sub_idx = df[df.model == model].index
    costs = np.column_stack([
        df.loc[sub_idx, "energy_mean_j"].values,     # Min Energy (J)
        df.loc[sub_idx, "cache_mib"].values,         # Min KV Cache (MiB)
        -df.loc[sub_idx, "tok_s_mean"].values,       # Max Decode tok/s
        -df.loc[sub_idx, "retrieval_estimate"].values # Max Hard Retrieval Rate
    ])
    eff_unconstrained = is_pareto_efficient_4d(costs)
    df.loc[sub_idx, "pareto_4d_unconstrained"] = eff_unconstrained
    
    # Constrained Pareto Analysis (Operational Viability: Quality >= 50%)
    valid_q = df.loc[sub_idx, "retrieval_estimate"].values >= 0.50
    costs_c = costs.copy()
    costs_c[~valid_q] = 1e9
    eff_constrained = is_pareto_efficient_4d(costs_c) & valid_q
    df.loc[sub_idx, "pareto_4d_constrained"] = eff_constrained

# 2,000 Paired Bootstrap Resampling in 4D space
np.random.seed(407)
N_BOOT = 2000

for model in ["llama", "qwen"]:
    sub_df = df[df.model == model].copy()
    sub_idx = sub_df.index
    roles = sub_df["normalized_role"].tolist()
    
    sub_e = e4096[e4096.model == model]
    sub_q = q4096[q4096.model == model]
    
    # Filter to 5 core roles
    sub_e_core = sub_e[sub_e.normalized_role.isin(roles)].copy()
    sub_q_core = sub_q[sub_q.normalized_role.isin(roles)].copy()
    
    # Build paired pivot matrices
    q_pivot = sub_q_core.pivot_table(index="scenario_id", columns="normalized_role", values="retrieval_correct", aggfunc="first").astype(float)
    e_pivot = sub_e_core.pivot_table(index="block", columns="normalized_role", values="total_energy_j", aggfunc="first")
    t_pivot = sub_e_core.pivot_table(index="block", columns="normalized_role", values="decode_tok_s", aggfunc="first")
    
    m_arr = np.array([sub_df[sub_df.normalized_role == r]["cache_mib"].iloc[0] for r in roles])
    
    counts_u = np.zeros(len(roles), dtype=int)
    counts_c = np.zeros(len(roles), dtype=int)
    
    q_mat = q_pivot[roles].values
    e_mat = e_pivot[roles].values
    t_mat = t_pivot[roles].values
    
    n_scenarios = len(q_mat)
    n_blocks = len(e_mat)
    
    for b in range(N_BOOT):
        idx_q = np.random.choice(n_scenarios, size=n_scenarios, replace=True)
        q_sample = np.mean(q_mat[idx_q], axis=0)
        
        idx_e = np.random.choice(n_blocks, size=n_blocks, replace=True)
        e_sample = np.mean(e_mat[idx_e], axis=0)
        t_sample = np.mean(t_mat[idx_e], axis=0)
        
        costs_b = np.column_stack([
            e_sample,
            m_arr,
            -t_sample,
            -q_sample
        ])
        
        eff_u = is_pareto_efficient_4d(costs_b)
        counts_u += eff_u.astype(int)
        
        valid = q_sample >= 0.50
        costs_c = costs_b.copy()
        costs_c[~valid] = 1e9
        eff_c = is_pareto_efficient_4d(costs_c) & valid
        counts_c += eff_c.astype(int)
        
    df.loc[sub_idx, "boot_freq_4d_unconstrained"] = (counts_u / N_BOOT) * 100
    df.loc[sub_idx, "boot_freq_4d_constrained"] = (counts_c / N_BOOT) * 100

df.to_csv(OUTPUT_DIR / "multidim_pareto_data.csv", index=False)
print("\n--- 4D Multi-Objective Pareto Analysis Results ---")
print(df[["model", "source_variant", "role_name", "energy_mean_j", "cache_mib", "tok_s_mean", "retrieval_estimate", "pareto_4d_unconstrained", "boot_freq_4d_unconstrained", "pareto_4d_constrained", "boot_freq_4d_constrained"]])

In [ ]:
# Run 3: Figure 1 - 3D Multi-Objective Pareto Space
fig = plt.figure(figsize=(12, 5.2))

for idx, model in enumerate(["llama", "qwen"]):
    ax = fig.add_subplot(1, 2, idx + 1, projection="3d")
    sub = df[df.model == model].copy()
    
    xs = sub["energy_mean_j"].values
    ys = sub["tok_s_mean"].values
    zs = sub["retrieval_estimate"].values
    
    # Drop lines to floor for visual depth perception
    for x, y, z in zip(xs, ys, zs):
        ax.plot([x, x], [y, y], [0, z], color="gray", linestyle=":", linewidth=0.8, alpha=0.6)
        
    for _, row in sub.iterrows():
        r_color = ROLE_COLORS[row["role_name"]]
        is_opt = row["pareto_4d_unconstrained"]
        lw = 2.4 if is_opt else 0.8
        ec = "black" if is_opt else "#666666"
        ax.scatter(
            row["energy_mean_j"], row["tok_s_mean"], row["retrieval_estimate"],
            s=60 + 0.35 * row["cache_mib"],
            color=r_color, edgecolor=ec, linewidth=lw, alpha=0.95, zorder=5
        )
        
        offset_z = 0.04 if row["retrieval_estimate"] > 0.3 else 0.08
        ax.text(
            row["energy_mean_j"], row["tok_s_mean"], row["retrieval_estimate"] + offset_z,
            f"{row['source_variant']}\n({row['role_name']})",
            fontsize=7.5, ha="center", va="bottom", weight="bold" if is_opt else "normal"
        )
        
    opt_sub = sub[sub.pareto_4d_unconstrained].sort_values("energy_mean_j")
    if len(opt_sub) >= 2:
        ax.plot(
            opt_sub["energy_mean_j"], opt_sub["tok_s_mean"], opt_sub["retrieval_estimate"],
            color="#2C3E50", linestyle="--", linewidth=1.5, alpha=0.85, label="4D Pareto Front"
        )
        
    if model == "llama" and len(opt_sub) >= 3:
        verts = [list(zip(opt_sub["energy_mean_j"], opt_sub["tok_s_mean"], opt_sub["retrieval_estimate"]))]
        poly = Poly3DCollection(verts, alpha=0.15, facecolor="#3498DB", edgecolor="#2C3E50", linestyle="--")
        ax.add_collection3d(poly)
        
    ax.set_title(MODEL_TITLES[model], pad=15, weight="bold")
    ax.set_xlabel("Total Energy (J) [Min $\\downarrow$]", labelpad=7)
    ax.set_ylabel("Decode Throughput (tok/s) [Max $\\uparrow$]", labelpad=7)
    ax.set_zlabel("Hard-Retrieval Rate [Max $\\uparrow$]", labelpad=7)
    ax.set_zlim(-0.05, 1.05)
    ax.view_init(elev=22, azim=-55 if model == "llama" else -50)

legend_handles = [Patch(facecolor=color, edgecolor="black", label=role) for role, color in ROLE_COLORS.items()]
legend_handles.append(plt.Line2D([0], [0], color="#2C3E50", linestyle="--", linewidth=1.5, label="4D Pareto Front"))
fig.legend(handles=legend_handles, loc="lower center", ncol=6, frameon=True, bbox_to_anchor=(0.5, -0.02))

fig.suptitle("Multi-Objective 4D Pareto Frontier (Energy $\\times$ Throughput $\\times$ Quality $\\times$ Cache Footprint)", y=0.98, weight="bold")
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.savefig(OUTPUT_DIR / "fig_pareto_4d_3d_scatter.pdf", bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "fig_pareto_4d_3d_scatter.png", bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
# Run 4: Figure 2 - 4-Axis Multi-Objective Parallel Coordinates Trade-Off Profile
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), sharey=True)

metrics = [
    ("energy_mean_j", "Total Energy (J)\n[Min $\\downarrow$]", True),
    ("cache_mib", "KV Cache (MiB)\n[Min $\\downarrow$]", True),
    ("tok_s_mean", "Throughput (tok/s)\n[Max $\\uparrow$]", False),
    ("retrieval_estimate", "Retrieval Rate (%)\n[Max $\\uparrow$]", False)
]

for ax_idx, model in enumerate(["llama", "qwen"]):
    ax = axes[ax_idx]
    sub = df[df.model == model].copy()
    norm_vals = np.zeros((len(sub), len(metrics)))
    for m_idx, (col, _, lower_is_better) in enumerate(metrics):
        g_min = df[col].min()
        g_max = df[col].max()
        if lower_is_better:
            norm_vals[:, m_idx] = (g_max - sub[col].values) / (g_max - g_min)
        else:
            norm_vals[:, m_idx] = (sub[col].values - g_min) / (g_max - g_min)
            
    x_coords = np.arange(len(metrics))
    for x in x_coords:
        ax.axvline(x, color="#D5D8DC", linestyle="-", linewidth=1.2, zorder=1)
        
    for i, (_, row) in enumerate(sub.iterrows()):
        r_color = ROLE_COLORS[row["role_name"]]
        is_opt = row["pareto_4d_unconstrained"]
        lw = 2.8 if is_opt else 1.3
        ls = "-" if is_opt else "--"
        alpha = 0.95 if is_opt else 0.45
        ax.plot(x_coords, norm_vals[i], color=r_color, linestyle=ls, linewidth=lw, alpha=alpha, zorder=3 if is_opt else 2)
        ax.scatter(
            x_coords, norm_vals[i],
            color=r_color,
            edgecolor="#1C2833" if is_opt else "#566573",
            linewidth=1.3 if is_opt else 0.8,
            s=68 if is_opt else 32,
            zorder=4 if is_opt else 3
        )
            
    ax.set_xticks(x_coords)
    ax.set_xticklabels([m[1] for m in metrics], weight="bold")
    ax.set_title(MODEL_TITLES[model], weight="bold", pad=12)
    ax.set_ylim(-0.05, 1.05)
    if ax_idx == 0:
        ax.set_ylabel("Normalized Efficiency Score (1.0 = Best Performance)", weight="bold")
    ax.grid(axis="y", linestyle=":", alpha=0.5)

color_handles = [Patch(facecolor=color, edgecolor="black", label=role) for role, color in ROLE_COLORS.items()]
style_handles = [
    plt.Line2D([0], [0], color="#2C3E50", linestyle="-", linewidth=2.5, marker="o", markersize=6, label="4D Pareto-Optimal (Solid)"),
    plt.Line2D([0], [0], color="#95A5A6", linestyle="--", linewidth=1.2, marker="o", markersize=4, label="Dominated (Dashed)")
]
all_legend = color_handles + style_handles

fig.legend(handles=all_legend, loc="lower center", ncol=7, frameon=True, bbox_to_anchor=(0.5, -0.05), fontsize=8.6)
fig.suptitle("4-Axis Parallel Coordinates Trade-Off Profile (Normalized Efficiency Comparison)", y=0.98, weight="bold")
plt.tight_layout(rect=[0, 0.06, 1, 0.95])
plt.savefig(OUTPUT_DIR / "fig_pareto_4d_parallel_coords.pdf", bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "fig_pareto_4d_parallel_coords.png", bbox_inches="tight")
plt.show()
plt.close()


In [ ]:
# Run 5: Figure 3 - Multi-Objective Radar / Spider Charts
categories = ["Energy Efficiency\n(1/Energy)", "Memory Efficiency\n(1/Cache)", "Decode Speed\n(Tokens/sec)", "Retrieval Fidelity\n(Accuracy)"]
N_cat = len(categories)
angles = [n / float(N_cat) * 2 * np.pi for n in range(N_cat)]
angles += angles[:1]

fig, axes = plt.subplots(1, 2, figsize=(11, 5.0), subplot_kw=dict(polar=True))

for idx, model in enumerate(["llama", "qwen"]):
    ax = axes[idx]
    sub = df[df.model == model].copy()
    
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=8.5, weight="bold")
    ax.set_ylim(0, 1.05)
    
    for _, row in sub.iterrows():
        r_color = ROLE_COLORS[row["role_name"]]
        is_opt = row["pareto_4d_unconstrained"]
        
        e_eff = df["energy_mean_j"].min() / row["energy_mean_j"]
        m_eff = df["cache_mib"].min() / row["cache_mib"]
        t_eff = row["tok_s_mean"] / df["tok_s_mean"].max()
        q_eff = row["retrieval_estimate"] / df["retrieval_estimate"].max()
        
        values = [e_eff, m_eff, t_eff, q_eff, e_eff]
        
        lw = 2.2 if is_opt else 1.0
        ls = "-" if is_opt else "--"
        alpha_fill = 0.22 if is_opt else 0.05
        
        ax.plot(angles, values, color=r_color, linewidth=lw, linestyle=ls, label=f"{row['source_variant']} ({row['role_name']})")
        ax.fill(angles, values, color=r_color, alpha=alpha_fill)
        
    ax.set_title(MODEL_TITLES[model], weight="bold", pad=20)

fig.legend(handles=legend_handles[:-1], loc="lower center", ncol=5, frameon=True, bbox_to_anchor=(0.5, -0.04))
fig.suptitle("Multi-Objective Efficiency Radar Profiles (Normalized to Best Observed System Metrics)", y=0.98, weight="bold")
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.savefig(OUTPUT_DIR / "fig_pareto_4d_radar.pdf", bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "fig_pareto_4d_radar.png", bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
# Run 6: Figure 4 - Pairwise 2D Multi-Objective Trade-Off Matrix
fig, axes = plt.subplots(2, 2, figsize=(10.5, 9.0))

pairs = [
    ("energy_mean_j", "retrieval_estimate", "Total Energy (J) [Min $\\downarrow$]", "Hard-Retrieval Rate [Max $\\uparrow$]", (0, 0)),
    ("energy_mean_j", "tok_s_mean", "Total Energy (J) [Min $\\downarrow$]", "Throughput (tok/s) [Max $\\uparrow$]", (0, 1)),
    ("cache_mib", "tok_s_mean", "Persistent KV Cache (MiB) [Min $\\downarrow$]", "Throughput (tok/s) [Max $\\uparrow$]", (1, 0)),
    ("cache_mib", "retrieval_estimate", "Persistent KV Cache (MiB) [Min $\\downarrow$]", "Hard-Retrieval Rate [Max $\\uparrow$]", (1, 1))
]

for x_col, y_col, x_lbl, y_lbl, (r, c) in pairs:
    ax = axes[r, c]
    
    for model, m_marker in [("llama", "o"), ("qwen", "s")]:
        sub = df[df.model == model].copy()
        
        for _, row in sub.iterrows():
            r_color = ROLE_COLORS[row["role_name"]]
            is_opt = row["pareto_4d_unconstrained"]
            ax.scatter(
                row[x_col], row[y_col],
                color=r_color, marker=m_marker, s=85 if is_opt else 45,
                edgecolor="black" if is_opt else "#666666", linewidth=1.5 if is_opt else 0.8,
                alpha=0.95 if is_opt else 0.6, zorder=4
            )
            ax.annotate(
                f"{row['source_variant']}", (row[x_col], row[y_col]),
                xytext=(4, 4), textcoords="offset points", fontsize=8, weight="bold" if is_opt else "normal"
            )
            
    ax.set_xlabel(x_lbl, weight="bold")
    ax.set_ylabel(y_lbl, weight="bold")
    ax.grid(True, linestyle=":", alpha=0.6)

marker_handles = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="gray", markersize=8, label="Llama 3.2-3B"),
    plt.Line2D([0], [0], marker="s", color="w", markerfacecolor="gray", markersize=8, label="Qwen2.5-3B")
]
all_handles = legend_handles[:-1] + marker_handles
fig.legend(handles=all_handles, loc="lower center", ncol=7, frameon=True, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Pairwise 2D Multi-Objective Projections and Trade-Off Planes", y=0.98, weight="bold")
plt.tight_layout(rect=[0, 0.04, 1, 0.96])
plt.savefig(OUTPUT_DIR / "fig_pareto_4d_tradeoff_matrix.pdf", bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "fig_pareto_4d_tradeoff_matrix.png", bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
# Run 7: Figure 5 - Master Composite Graphic (ACM Sigconf 2-Column Publication Format)
fig = plt.figure(figsize=(13, 9.5))
gs = gridspec.GridSpec(2, 2, height_ratios=[1.15, 1.0], hspace=0.32, wspace=0.22)

# (a) Llama 3D
ax_llama3d = fig.add_subplot(gs[0, 0], projection="3d")
sub_l = df[df.model == "llama"].copy()
xs, ys, zs = sub_l["energy_mean_j"], sub_l["tok_s_mean"], sub_l["retrieval_estimate"]
for x, y, z in zip(xs, ys, zs):
    ax_llama3d.plot([x, x], [y, y], [0, z], color="gray", linestyle=":", linewidth=0.8, alpha=0.6)
for _, row in sub_l.iterrows():
    is_opt = row["pareto_4d_unconstrained"]
    ax_llama3d.scatter(
        row["energy_mean_j"], row["tok_s_mean"], row["retrieval_estimate"],
        s=55 + 0.35 * row["cache_mib"], color=ROLE_COLORS[row["role_name"]],
        edgecolor="black" if is_opt else "#666666", linewidth=2.2 if is_opt else 0.8, zorder=5
    )
    ax_llama3d.text(
        row["energy_mean_j"], row["tok_s_mean"], row["retrieval_estimate"] + 0.04,
        f"{row['source_variant']}", fontsize=7.5, ha="center", weight="bold" if is_opt else "normal"
    )
opt_l = sub_l[sub_l.pareto_4d_unconstrained].sort_values("energy_mean_j")
ax_llama3d.plot(opt_l["energy_mean_j"], opt_l["tok_s_mean"], opt_l["retrieval_estimate"], color="#2C3E50", linestyle="--", linewidth=1.5)
verts_l = [list(zip(opt_l["energy_mean_j"], opt_l["tok_s_mean"], opt_l["retrieval_estimate"]))]
ax_llama3d.add_collection3d(Poly3DCollection(verts_l, alpha=0.15, facecolor="#3498DB", edgecolor="#2C3E50", linestyle="--"))
ax_llama3d.set_title("(a) Llama 3.2-3B: 3D Pareto Space", weight="bold")
ax_llama3d.set_xlabel("Energy (J) [$\\downarrow$]", labelpad=5, fontsize=8.5)
ax_llama3d.set_ylabel("Tok/s [$\\uparrow$]", labelpad=5, fontsize=8.5)
ax_llama3d.set_zlabel("Quality [$\\uparrow$]", labelpad=5, fontsize=8.5)
ax_llama3d.set_zlim(-0.05, 1.05)
ax_llama3d.view_init(elev=22, azim=-55)

# (b) Qwen 3D
ax_qwen3d = fig.add_subplot(gs[0, 1], projection="3d")
sub_q = df[df.model == "qwen"].copy()
xs, ys, zs = sub_q["energy_mean_j"], sub_q["tok_s_mean"], sub_q["retrieval_estimate"]
for x, y, z in zip(xs, ys, zs):
    ax_qwen3d.plot([x, x], [y, y], [0, z], color="gray", linestyle=":", linewidth=0.8, alpha=0.6)
for _, row in sub_q.iterrows():
    is_opt = row["pareto_4d_unconstrained"]
    ax_qwen3d.scatter(
        row["energy_mean_j"], row["tok_s_mean"], row["retrieval_estimate"],
        s=55 + 0.35 * row["cache_mib"], color=ROLE_COLORS[row["role_name"]],
        edgecolor="black" if is_opt else "#666666", linewidth=2.2 if is_opt else 0.8, zorder=5
    )
    ax_qwen3d.text(
        row["energy_mean_j"], row["tok_s_mean"], row["retrieval_estimate"] + 0.05,
        f"{row['source_variant']}", fontsize=7.5, ha="center", weight="bold" if is_opt else "normal"
    )
opt_q = sub_q[sub_q.pareto_4d_unconstrained].sort_values("energy_mean_j")
ax_qwen3d.plot(opt_q["energy_mean_j"], opt_q["tok_s_mean"], opt_q["retrieval_estimate"], color="#2C3E50", linestyle="--", linewidth=1.5)
ax_qwen3d.set_title("(b) Qwen2.5-3B: 3D Pareto Space", weight="bold")
ax_qwen3d.set_xlabel("Energy (J) [$\\downarrow$]", labelpad=5, fontsize=8.5)
ax_qwen3d.set_ylabel("Tok/s [$\\uparrow$]", labelpad=5, fontsize=8.5)
ax_qwen3d.set_zlabel("Quality [$\\uparrow$]", labelpad=5, fontsize=8.5)
ax_qwen3d.set_zlim(-0.05, 1.05)
ax_qwen3d.view_init(elev=22, azim=-50)

# (c) Parallel Coordinates
ax_pc = fig.add_subplot(gs[1, 0])
for model, m_marker, m_ls in [("llama", "o", "-"), ("qwen", "s", "--")]:
    sub = df[df.model == model].copy()
    norm_vals = np.zeros((len(sub), len(metrics)))
    for m_idx, (col, _, lower_is_better) in enumerate(metrics):
        g_min, g_max = df[col].min(), df[col].max()
        if lower_is_better:
            norm_vals[:, m_idx] = (g_max - sub[col].values) / (g_max - g_min)
        else:
            norm_vals[:, m_idx] = (sub[col].values - g_min) / (g_max - g_min)
    for i, (_, row) in enumerate(sub.iterrows()):
        is_opt = row["pareto_4d_unconstrained"]
        ax_pc.plot(np.arange(4), norm_vals[i], color=ROLE_COLORS[row["role_name"]], linestyle=m_ls, linewidth=2.0 if is_opt else 1.0, alpha=0.9 if is_opt else 0.4)
        ax_pc.scatter(np.arange(4), norm_vals[i], color=ROLE_COLORS[row["role_name"]], marker=m_marker, edgecolor="black" if is_opt else "#555555", s=40 if is_opt else 20)
for x in range(4):
    ax_pc.axvline(x, color="#DDDDDD", linestyle="-", linewidth=0.9, zorder=1)
ax_pc.set_xticks(range(4))
ax_pc.set_xticklabels([m[1] for m in metrics], weight="bold", fontsize=8)
ax_pc.set_ylabel("Normalized Efficiency (1.0 = Best)", weight="bold")
ax_pc.set_title("(c) 4-Axis Parallel Coordinates Trade-Offs", weight="bold")
ax_pc.grid(axis="y", linestyle=":", alpha=0.6)

# (d) Radar Envelopes
ax_rad = fig.add_subplot(gs[1, 1], polar=True)
ax_rad.set_theta_offset(np.pi / 2)
ax_rad.set_theta_direction(-1)
ax_rad.set_xticks(angles[:-1])
ax_rad.set_xticklabels(categories, size=7.8, weight="bold")
ax_rad.set_ylim(0, 1.05)

for _, row in df[df.pareto_4d_unconstrained].iterrows():
    r_color = ROLE_COLORS[row["role_name"]]
    m_ls = "-" if row["model"] == "llama" else "--"
    e_eff = df["energy_mean_j"].min() / row["energy_mean_j"]
    m_eff = df["cache_mib"].min() / row["cache_mib"]
    t_eff = row["tok_s_mean"] / df["tok_s_mean"].max()
    q_eff = row["retrieval_estimate"] / df["retrieval_estimate"].max()
    vals = [e_eff, m_eff, t_eff, q_eff, e_eff]
    ax_rad.plot(angles, vals, color=r_color, linewidth=1.8, linestyle=m_ls, label=f"{row['source_variant']} ({row['role_name']})")
    ax_rad.fill(angles, vals, color=r_color, alpha=0.15)
ax_rad.set_title("(d) Non-Dominated Frontier Envelopes", weight="bold", pad=15)

fig.legend(handles=all_handles, loc="lower center", ncol=7, frameon=True, bbox_to_anchor=(0.5, -0.02), fontsize=8.5)
fig.suptitle("Multi-Dimensional Multi-Objective Pareto Frontier and Systems Trade-Offs on NVIDIA T4", y=0.98, weight="bold", fontsize=12)
plt.savefig(OUTPUT_DIR / "fig_pareto_4d_composite_master.pdf", bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "fig_pareto_4d_composite_master.png", bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
# Run 8: Interactive 3D/4D Plotly HTML Explorer
if HAS_PLOTLY:
    fig_plotly = make_subplots(
        rows=1, cols=2,
        specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
        subplot_titles=(MODEL_TITLES["llama"], MODEL_TITLES["qwen"])
    )

    for col_idx, model in enumerate(["llama", "qwen"], start=1):
        sub = df[df.model == model].copy()
        for _, row in sub.iterrows():
            hover_text = (
                f"<b>Variant: {row['source_variant']} ({row['role_name']})</b><br>"
                f"• Total Energy: {row['energy_mean_j']:.2f} ± {row['energy_sd_j']:.2f} J<br>"
                f"• Persistent KV Cache: {row['cache_mib']:.2f} MiB ({int(row['persistent_kv_bytes_mean']):,} bytes)<br>"
                f"• Decode Throughput: {row['tok_s_mean']:.2f} ± {row['tok_s_std']:.2f} tok/s<br>"
                f"• Hard Retrieval Accuracy: {row['retrieval_estimate']*100:.1f}% ({int(row['retrieval_successes'])}/60)<br>"
                f"• 4D Pareto Optimal: <b>{'YES' if row['pareto_4d_unconstrained'] else 'NO'}</b><br>"
                f"• Bootstrap 4D Stability: {row['boot_freq_4d_unconstrained']:.1f}%"
            )
            fig_plotly.add_trace(
                go.Scatter3d(
                    x=[row["energy_mean_j"]],
                    y=[row["tok_s_mean"]],
                    z=[row["retrieval_estimate"]],
                    mode="markers+text",
                    marker=dict(
                        size=max(8, row["cache_mib"] / 20),
                        color=ROLE_COLORS[row["role_name"]],
                        line=dict(color="black", width=3 if row["pareto_4d_unconstrained"] else 1),
                        opacity=0.9
                    ),
                    text=[f"{row['source_variant']}"],
                    textposition="top center",
                    name=f"{row['source_variant']} ({row['role_name']})",
                    hoverinfo="text",
                    hovertext=hover_text,
                    showlegend=(col_idx == 1)
                ),
                row=1, col=col_idx
            )
        
        opt_sub = sub[sub.pareto_4d_unconstrained].sort_values("energy_mean_j")
        if len(opt_sub) >= 2:
            fig_plotly.add_trace(
                go.Scatter3d(
                    x=opt_sub["energy_mean_j"],
                    y=opt_sub["tok_s_mean"],
                    z=opt_sub["retrieval_estimate"],
                    mode="lines",
                    line=dict(color="#2C3E50", width=4, dash="dash"),
                    name="4D Pareto Frontier",
                    hoverinfo="none",
                    showlegend=(col_idx == 1)
                ),
                row=1, col=col_idx
            )

    fig_plotly.update_layout(
        title="Interactive 4D Multi-Objective Pareto Explorer (NVIDIA T4 SLM Compression)",
        scene=dict(xaxis_title="Energy (J) [Min]", yaxis_title="Throughput (tok/s) [Max]", zaxis_title="Retrieval Rate [Max]"),
        scene2=dict(xaxis_title="Energy (J) [Min]", yaxis_title="Throughput (tok/s) [Max]", zaxis_title="Retrieval Rate [Max]"),
        height=650, width=1200
    )
    fig_plotly.write_html(OUTPUT_DIR / "interactive_pareto_4d.html")
    print(f"Interactive Plotly dashboard saved to: {OUTPUT_DIR / 'interactive_pareto_4d.html'}")
else:
    print("Plotly is not installed; skipping interactive HTML export.")

In [ ]:
# Run 9: LaTeX Table Generation and Markdown Summary Export
tex_content = r"""\begin{table*}[htbp]
\caption{Multi-Dimensional 4-Objective Pareto Frontier Analysis at 4096 Tokens}
\label{tab:pareto4d}
\centering\small
\begin{tabular}{@{}llrrrrccc@{}}
\toprule
\textbf{Model} & \textbf{Config} & \textbf{Energy (J) $\downarrow$} & \textbf{Cache (MiB) $\downarrow$} & \textbf{Throughput (tok/s) $\uparrow$} & \textbf{Retrieval $\uparrow$} & \textbf{4D Frontier} & \textbf{Boot.\% (4D)} & \textbf{Constrained ($Q \ge 50\%$)} \\
\midrule
"""

for model in ["llama", "qwen"]:
    sub = df[df.model == model]
    m_name = "Llama 3.2-3B" if model == "llama" else "Qwen2.5-3B"
    for idx_r, (_, row) in enumerate(sub.iterrows()):
        mod_prefix = m_name if idx_r == 0 else ""
        opt_str = r"\textbf{Yes}" if row["pareto_4d_unconstrained"] else "No"
        c_str = r"\textbf{Yes}" if row["pareto_4d_constrained"] else (r"\xmark\ (Failed $Q$)" if row["retrieval_estimate"] < 0.50 else "No (Dominated)")
        tex_content += f"{mod_prefix:14s} & {row['source_variant']:8s} & {row['energy_mean_j']:.2f} $\\pm$ {row['energy_sd_j']:.2f} & {row['cache_mib']:.1f} & {row['tok_s_mean']:.2f} $\\pm$ {row['tok_s_std']:.2f} & {row['retrieval_estimate']:.3f} ({int(row['retrieval_successes'])}/60) & {opt_str} & {row['boot_freq_4d_unconstrained']:.1f}\\% & {c_str} \\\\
"
    if model == "llama":
        tex_content += r"\midrule" + "\n"

tex_content += r"""\bottomrule
\end{tabular}
\begin{minipage}{\textwidth}\vspace{3pt}\footnotesize
\textbf{Objectives}: Minimize total energy ($J$), minimize persistent KV cache footprint ($\text{MiB}$), maximize decode throughput ($\text{tok/s}$), and maximize hard-retrieval accuracy ($Q$).\\
\textbf{Boot.\% (4D)}: Percentage of 2000 paired bootstrap resamples where the configuration forms the 4D empirical Pareto frontier.\\
\textbf{Constrained ($Q \ge 50\%$)}: Pareto optimality evaluated subject to an operational viability constraint requiring at least 50\% retrieval success.
\end{minipage}
\end{table*}
"""

with open(OUTPUT_DIR / "multidim_pareto_table.tex", "w") as f:
    f.write(tex_content)

# Markdown Summary
md_summary = f"""# 4-Dimensional Multi-Objective Pareto Analysis Summary

## Key Findings
1. **Llama 3.2-3B (3:1 GQA)**:
   - **C2 (AWQ + Eviction)** is non-dominated across all 4 dimensions: minimal total energy (529.30 J), smallest cache footprint (119.0 MiB), highest throughput (20.34 tok/s), and high retrieval quality (37/60 = 61.7%). 4D Bootstrap Selection: **{df[(df.model=='llama') & (df.source_variant=='C2')]['boot_freq_4d_unconstrained'].iloc[0]:.1f}%**.
   - **C0 (FP16)** is non-dominated due to maximal retrieval quality (47/60 = 78.3%).
   - **C1-W (AWQ)** is non-dominated due to high retrieval (38/60) with moderate energy (639.86 J).
   - **C3-Ref** (913.48 J, 696.5 MiB, 7.66 tok/s) and **C4-Ref** (659.50 J, 187.2 MiB, 12.53 tok/s) are dominated by C2 across energy, memory, and throughput without improving accuracy.

2. **Qwen2.5-3B (8:1 GQA)**:
   - **Q0 (FP16)** is the single viable Pareto-optimal operating point under operational quality constraints (616.78 J, 146.2 MiB, 13.23 tok/s, 53/60 = 88.3%).
   - **Q2 (AWQ + Eviction)** achieves the lowest energy (550.48 J) and lowest memory (38.2 MiB), but catastrophically fails retrieval (1/60 = 1.7%), making it non-viable for practical deployment.
   - **Q3-Ref** preserves retrieval (49/60 = 81.7%) but incurs heavy energy (812.28 J), memory (223.9 MiB), and throughput penalties (8.60 tok/s).
"""

with open(OUTPUT_DIR / "multidim_pareto_summary.md", "w") as f:
    f.write(md_summary)

print("LaTeX table and Markdown summary exported successfully.")

In [ ]:
# Run 10: Artifact Integrity Audit, Manifest Generation, and Downloadable ZIP Packaging
manifest = {
    "schema": "cse407-multidim-pareto-v1-2026-08-20-1",
    "objectives": ["total_energy_j (min)", "cache_mib (min)", "decode_tok_s (max)", "retrieval_rate (max)"],
    "context": 4096,
    "bootstrap_resamples": N_BOOT,
    "files": {}
}

for p in sorted(OUTPUT_DIR.glob("*")):
    if p.is_file() and p.suffix != ".zip":
        with open(p, "rb") as f:
            manifest["files"][p.name] = {
                "size_bytes": p.stat().st_size,
                "sha256": hashlib.sha256(f.read()).hexdigest()
            }

with open(OUTPUT_DIR / "output_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

# Package downloadable ZIP archive
zip_out = OUTPUT_DIR.parent / "CSE407_multidim_pareto_v1.zip"
with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as zipf:
    for p in OUTPUT_DIR.glob("*"):
        if p.is_file() and p.name != "CSE407_multidim_pareto_v1.zip":
            zipf.write(p, arcname=p.name)

print("=" * 60)
print(f"ALL ARTIFACTS PACKAGED SUCCESSFULLY!")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Downloadable ZIP: {zip_out} ({zip_out.stat().st_size / 1e3:.1f} KB)")
print(f"Total artifacts: {len(manifest['files'])}")
print("=" * 60)